# Resolving Conflicts by Encoding the Precedences into the Grammar

The notebook [01-Conflicts.ipynb](01-Conflicts.ipynb) has shown that the following grammar is *ambiguous*,
because it does not specify the precedences of the arithmetical operators:
```
    expr : expr "+" expr
         | expr "-" expr
         | expr "*" expr
         | expr "/" expr
         | expr "^" expr
         | "(" expr ")"
         | NUMBER
         ;
```

`Larkita` can resolve the resulting *shift/reduce conflicts* with *precedence declarations* of the form
```
    %left  "+" "-";      // precedence 1
    %left  "*" "/";      // precedence 2
    %right "^";          // precedence 3
```
as the notebook [03-Precedence-Declarations.ipynb](03-Precedence-Declarations.ipynb) shows.  In this notebook, we
take the other approach:  We encode the precedences and associativities of the operators into the grammar itself.

## Encoding the Precedences

The recipe is the one that we have already used in the chapter on context-free languages:  we introduce one
syntactical variable $v_n$ for every precedence level $n$ and define the variable of a given precedence level
in terms of the variable of the next higher precedence level.  Let $o$ be an operator of precedence level $n$.

* If $o$ is **left associative**, the corresponding rule is **left recursive**:
  $$ v_n \rightarrow v_n \; o \; v_{n+1} $$
* If $o$ is **right associative**, the rule is **right recursive**:
  $$ v_n \rightarrow v_{n+1} \; o \; v_n $$
* If $o$ is **non-associative**, the rule uses neither recursion:
  $$ v_n \rightarrow v_{n+1} \; o \; v_{n+1} $$

For the grammar above we need three precedence levels plus one variable for the expressions that bind most
tightly:

| variable | precedence | operators | associativity |
|:---------|:-----------|:----------|:--------------|
| `expr`   | 1          | `+`, `-`  | left          |
| `prod`   | 2          | `*`, `/`  | left          |
| `fact`   | 3          | `^`       | right         |
| `atom`   | &mdash;    | &mdash;   | &mdash;       |

Note how the associativities show up in the grammar below:  the rules for `expr` and `prod` are *left*
recursive, while the rule `fact : atom "^" fact;` is *right* recursive.

As we have discussed already previously when introducing `Larkita`, the abstract syntax tree that
`Larkita` builds is determined by the grammar:  The string literals of an alternative
form the *head* of the tuple that is returned for this alternative, while all other symbols become its children.

1. Hence the alternative `expr "+" prod` yields the tuple `('+', lhs, rhs)`.
2. An alternative that consists of a single symbol, like `expr : prod`, returns the abstract syntax
   tree of this symbol.
   Without this, every number would be buried under a chain `expr` &rarr; `prod` &rarr; `fact` &rarr; `atom`
   that carries no information.
4. The annotation `%ignore "(", ")"` at the end of the alternative `"(" expr ")"` removes the parentheses from
   its head.  Then this alternative consists of a single symbol and returns the abstract syntax tree of `expr`.

In [1]:
from IPython.display import Code, display

Code(filename='conflicts-resolved.lark', language='larkita')

expr : expr "+" prod
     | expr "-" prod
     | prod
     ;

prod : prod "*" fact
     | prod "/" fact
     | fact
     ;

fact : atom "^" fact
     | atom
     ;

atom : "(" expr ")"    %ignore "(", ")"
     | NUMBER
     ;

NUMBER : /0|[1-9][0-9]*/;

%import common.WS;
%ignore WS;

## Specification of the Parser

For a grammar with shift-reduce conflicts, `Larkita` issues a `ConflictWarning`, which `Jupyter` shows below the
cell that creates the parser.  *Reduce-reduce conflicts* raise an exception of class `GrammarError`.

Since no warning is shown below and the parser has no conflicts, our grammar is an `LR(1)` grammar.

In [2]:
from larkita import Larkita

parser = Larkita.open('conflicts-resolved.lark', out_file='conflicts-resolved.out')

In [3]:
print(parser.conflicts)

[]


## Inspecting the LR(1) States

`Larkita` writes the states of the canonical LR(1) automaton into the file given by the keyword argument `out_file`.
Every state lists its *marked rules* together with their *lookahead* tokens, followed by the actions and the gotos of
the state.  The marked rules that are added by the `closure` operation are marked with a `+`.

`Larkita` marks the position of the parser inside a rule with the bullet `•` that we use in the lecture notes, 
and the end of the input is called `$END`.

Note that there is no state that offers both a `shift` and a `reduce` action for the same token:  all
conflicts are gone.

In [4]:
with open('conflicts-resolved.out') as handle:
    print(handle.read())

Canonical LR(1) automaton generated by Larkita

Grammar:      conflicts-resolved.lark
Start symbol: expr
States:       36
Conflicts:    0 (0 unresolved)


Rules
=====

Every rule is followed by the abstract syntax tree that a reduction produces.
"*name" stands for the list of children collected by the auxiliary rule "name".
"text" stands for the text of a named token.

  (0)   $root_expr → expr
  (1)   expr → expr "+" prod  ⇒ ("+", ast(expr), ast(prod))
  (2)   expr → expr "-" prod  ⇒ ("-", ast(expr), ast(prod))
  (3)   expr → prod           ⇒ ast(prod)
  (4)   prod → prod "*" fact  ⇒ ("*", ast(prod), ast(fact))
  (5)   prod → prod "/" fact  ⇒ ("/", ast(prod), ast(fact))
  (6)   prod → fact           ⇒ ast(fact)
  (7)   fact → atom "^" fact  ⇒ ("^", ast(atom), ast(fact))
  (8)   fact → atom           ⇒ ast(atom)
  (9)   atom → "(" expr ")"   ⇒ ast(expr)
  (10)  atom → NUMBER         ⇒ ("NUMBER", text)


Conflicts

There are no conflicts.


States

Every state lists its items with their